# Phase 2.1 - More Analysis

In [2]:
# Spark session
import os
from pyspark.sql import SparkSession

# Set JAVA_HOME and PATH
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk"
os.environ["PATH"] = f"/usr/lib/jvm/java-17-openjdk-amd64/bin:{os.environ['PATH']}"

spark = SparkSession.builder.getOrCreate()

### Merged data

In [13]:
mdf = spark.read.option("header", True).option("inferSchema", True).csv("data/merged_data.csv")

print("Unified dataset info:")
print(f"Total rows: {mdf.count()}")
mdf.printSchema()
mdf.show(5)

Unified dataset info:
Total rows: 75663
root
 |-- lead_time: integer (nullable = true)
 |-- market_segment_type: string (nullable = true)
 |-- avg_price_per_room: double (nullable = true)
 |-- booking_status: integer (nullable = true)
 |-- arrival_date: date (nullable = true)
 |-- total_stay_nights: integer (nullable = true)

+---------+-------------------+------------------+--------------+------------+-----------------+
|lead_time|market_segment_type|avg_price_per_room|booking_status|arrival_date|total_stay_nights|
+---------+-------------------+------------------+--------------+------------+-----------------+
|       56|            Offline|             120.0|             0|  2018-06-08|                1|
|        2|             Online|             197.0|             0|  2018-05-27|                1|
|       36|             Online|              88.0|             1|  2018-12-08|                3|
|       21|             Online|            124.25|             0|  2018-08-29|            

### Cancellation rates by month

In [32]:
from pyspark.sql.functions import month, year, sum as ssum, when, col, count, round

# merged dataframe with a month column for easier monthly analysis
mdf_month = mdf.withColumn("arrival_month", month("arrival_date"))

cancellation_rates = mdf_month.groupBy("arrival_month").agg(
        count("*").alias("total_bookings"),
        ssum("booking_status").alias("cancellations"),
        round((ssum("booking_status") / count("*")) * 100, 2).alias("cancellation_rate")
    ).orderBy("arrival_month")

print("Cancellation rates by month")
cancellation_rates.show(12)

Cancellation rates by month
+-------------+--------------+-------------+-----------------+
|arrival_month|total_bookings|cancellations|cancellation_rate|
+-------------+--------------+-------------+-----------------+
|            1|          2408|          301|             12.5|
|            2|          3838|          782|            20.38|
|            3|          5444|         1393|            25.59|
|            4|          5400|         1587|            29.39|
|            5|          5103|         1454|            28.49|
|            6|          4952|         1468|            29.64|
|            7|          7470|         2454|            32.85|
|            8|          9427|         3150|            33.41|
|            9|          8792|         2406|            27.37|
|           10|          9187|         2442|            26.58|
|           11|          6640|         1486|            22.38|
|           12|          7002|         1646|            23.51|
+-------------+------------

We group the arrival months and get the total number of bookings(rows), creating a "total_bookings" column. We then count the cancellations, because we made "Canceled" to be 1 we can sum the column to get the total number of cancellations. Then we can create a cancellation rate column with those new columns.

### Averages

In [15]:
monthly_avgs = mdf_month.groupBy("arrival_month").agg(
        round(ssum("avg_price_per_room") / count("*"), 2).alias("avg_price_per_room"),
        round(ssum("total_stay_nights") / count("*"), 2).alias("avg_stay_nights"),
        count("*").alias("total_bookings")
    ).orderBy("arrival_month")

print("Monthly averages for price and nights")
monthly_avgs.show(12)

Monthly averages for price and nights
+-------------+------------------+---------------+--------------+
|arrival_month|avg_price_per_room|avg_stay_nights|total_bookings|
+-------------+------------------+---------------+--------------+
|            1|             66.99|           2.92|          2408|
|            2|              74.6|           3.06|          3838|
|            3|             84.69|           3.25|          5444|
|            4|             97.21|           3.27|          5400|
|            5|            107.43|           3.32|          5103|
|            6|            111.81|           3.52|          4952|
|            7|            123.31|           4.12|          7470|
|            8|            133.94|           3.98|          9427|
|            9|            115.65|           3.59|          8792|
|           10|             97.58|           3.27|          9187|
|           11|             79.53|           3.32|          6640|
|           12|             85.67|    

Here we do very similar operations to the last cell, grouping by each month, and creating columns for the averages for that month specifically.

### Monthly Bookings

In [31]:
monthly_seg_bookings = mdf_month.groupBy("arrival_month", "market_segment_type").agg(
    count("*").alias("booking_count")
).orderBy("arrival_month", "market_segment_type")

print("Monthly bookings by market segment")
monthly_seg_bookings.show(106)

Monthly bookings by market segment
+-------------+-------------------+-------------+
|arrival_month|market_segment_type|booking_count|
+-------------+-------------------+-------------+
|            1|      Complementary|           44|
|            1|          Corporate|          200|
|            1|             Direct|          324|
|            1|             Groups|           19|
|            1|            Offline|          152|
|            1|      Offline TA/TO|          308|
|            1|             Online|          513|
|            1|          Online TA|          848|
|            2|           Aviation|            2|
|            2|      Complementary|           39|
|            2|          Corporate|          292|
|            2|             Direct|          451|
|            2|             Groups|           45|
|            2|            Offline|          228|
|            2|      Offline TA/TO|          499|
|            2|             Online|          919|
|            2|

Here we group the arrival months and the market segment types and create a total_bookings column for each.

### Seasonality

In [36]:
seasonal_df = mdf_month.withColumn("estimated_revenue", col("avg_price_per_room") * col("total_stay_nights"))

monthly_rev = seasonal_df.groupBy("arrival_month").agg(
    round(ssum("estimated_revenue"), 2).alias("total_revenue"),
    count("*").alias("total_bookings"),
    round(ssum("estimated_revenue") / count("*"), 2).alias("avg_revenue_per_booking")
).orderBy("total_revenue", ascending=False)

print("Seasonality analysis - Monthly revenue")
monthly_rev.show(12)

Seasonality analysis - Monthly revenue
+-------------+-------------+--------------+-----------------------+
|arrival_month|total_revenue|total_bookings|avg_revenue_per_booking|
+-------------+-------------+--------------+-----------------------+
|            8|   5125856.02|          9427|                 543.74|
|            7|   3799882.77|          7470|                 508.69|
|            9|   3503063.36|          8792|                 398.44|
|           10|   2893744.81|          9187|                 314.98|
|           12|    2037449.7|          7002|                 290.98|
|            6|   1906939.22|          4952|                 385.08|
|           11|    1762126.8|          6640|                 265.38|
|            5|   1757515.69|          5103|                 344.41|
|            4|   1706121.88|          5400|                 315.95|
|            3|   1484732.86|          5444|                 272.73|
|            2|    882224.98|          3838|                 229

Here we create an "estimated_revenue" column by taking the avg_price_per_room and multiplying it with the total_stay_nights.

Then we group by the arrival month again, and update the estimated_revenue to be the total_revenue, and average the estimated revenue per month into "avg_revenue_per_booking", sorting by the total revenue in the end.